In [1]:
from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path(
    r"D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI"
)

PROCESSED_DIR = BASE_DIR / "datasets" / "processed"
REPORT_DIR = BASE_DIR / "datasets" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Processed folder exists:", PROCESSED_DIR.exists())

Processed folder exists: True


In [2]:
expected_files = [
    "perishable_inventory_clean.csv",
    "inventory_batches_clean.csv",
    "discount_scenarios_simulated_clean.csv",
    "calendar_clean.csv",
    "sample_submission_validated.csv"
]

for filename in expected_files:
    file_path = PROCESSED_DIR / filename

    print(
        filename,
        "->",
        "FOUND" if file_path.exists() else "NOT FOUND"
    )

perishable_inventory_clean.csv -> FOUND
inventory_batches_clean.csv -> FOUND
discount_scenarios_simulated_clean.csv -> FOUND
calendar_clean.csv -> FOUND
sample_submission_validated.csv -> FOUND


In [3]:
products_path = PROCESSED_DIR / "products_clean.csv"

if not products_path.exists():
    raise FileNotFoundError(
        f"File not found: {products_path}"
    )

products_df = pd.read_csv(products_path)

print("Products shape:", products_df.shape)
print("Product columns:", products_df.columns.tolist())

display(products_df.head())

Products shape: (10, 6)
Product columns: ['product_id', 'product_name', 'category', 'shelf_life_days', 'selling_price', 'unit_cost']


,product_id,product_name,category,shelf_life_days,selling_price,unit_cost
0,1001,Milk 500ml,Dairy,5,32.0,24.0
1,1002,Yogurt Cup,Dairy,10,25.0,17.0
2,1003,Bread Loaf,Bakery,4,40.0,26.0
3,1004,Paneer 200g,Dairy,7,85.0,62.0
4,1005,Spinach Pack,Produce,3,30.0,18.0


In [4]:
perishable_path = PROCESSED_DIR / "perishable_inventory_clean.csv"
batches_path = PROCESSED_DIR / "inventory_batches_clean.csv"

perishable_df = pd.read_csv(perishable_path)
batches_df = pd.read_csv(batches_path)

print("Perishable inventory:", perishable_df.shape)
print("Inventory batches:", batches_df.shape)

Perishable inventory: (1800, 12)
Inventory batches: (2000, 16)


In [5]:
product_ids = set(
    products_df["product_id"].astype(str).str.strip()
)

perishable_product_ids = set(
    perishable_df["product_id"].astype(str).str.strip()
)

batch_product_ids = set(
    batches_df["product_id"].astype(str).str.strip()
)

print(
    "Perishable IDs not found in products:",
    len(perishable_product_ids - product_ids)
)

print(
    "Batch product IDs not found in products:",
    len(batch_product_ids - product_ids)
)

Perishable IDs not found in products: 0
Batch product IDs not found in products: 0


In [6]:
discount_path = (
    PROCESSED_DIR / "discount_scenarios_simulated_clean.csv"
)

discount_df = pd.read_csv(discount_path)

print("Discount scenarios shape:", discount_df.shape)
print("Discount columns:", discount_df.columns.tolist())

if "batch_id" in discount_df.columns:
    known_batch_ids = set(
        batches_df["batch_id"].astype(str).str.strip()
    )

    scenario_batch_ids = set(
        discount_df["batch_id"].astype(str).str.strip()
    )

    print(
        "Scenario batch IDs not found in inventory:",
        len(scenario_batch_ids - known_batch_ids)
    )
else:
    print("No batch_id column; inspect the columns before checking joins.")

Discount scenarios shape: (10000, 8)
Discount columns: ['batch_id', 'product_id', 'days_to_expiry', 'quantity_remaining', 'discount_pct', 'estimated_units_sold_simulated', 'estimated_revenue_simulated', 'estimated_gross_margin_simulated']
Scenario batch IDs not found in inventory: 0


In [7]:
report = {
    "products_rows": int(len(products_df)),
    "perishable_inventory_rows": int(len(perishable_df)),
    "inventory_batches_rows": int(len(batches_df)),
    "perishable_product_ids_not_found": int(
        len(perishable_product_ids - product_ids)
    ),
    "batch_product_ids_not_found": int(
        len(batch_product_ids - product_ids)
    )
}

report_path = REPORT_DIR / "dataset_relationship_validation.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=4)

print("Validation report saved to:", report_path)

Validation report saved to: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports\dataset_relationship_validation.json
